In [ ]:
!pip -q install timm segmentation-models-pytorch fastai

from fastai.vision.all import *
from pathlib import Path
import segmentation_models_pytorch as smp
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:

DATA_ROOT = Path("dataset")     
IMAGES_DIR = DATA_ROOT/'images'
MASKS_DIR  = DATA_ROOT/'masks'  

print("Images dir:", IMAGES_DIR.resolve())
print("Masks  dir:", MASKS_DIR.resolve())

# quick sanity
images = get_image_files(IMAGES_DIR)
labels = get_image_files(MASKS_DIR)
print(f"Found {len(images)} images, {len(labels)} masks")
assert len(images) > 0, "No images found — check IMAGES_DIR"
assert len(labels) > 0, "No masks found — check MASKS_DIR"
images[:3], labels[:3]


In [ ]:
im = images[2400]; im

In [ ]:
PILImage.create(im)

In [ ]:
labels = get_image_files(MASKS_DIR)
labels

In [ ]:
lbl = labels[2400]; lbl

In [ ]:
msk = PILMask.create(lbl); msk
np.unique(msk)

In [ ]:
RectalCodes = (
    'Background',
    'RectalWall',
    'Lesion',
    'RectalWallDefect',
    'Lumen',
    'Rectoscope',
    'Retractor',
    'Diathermy',
    'Scissors',
    'Sucker',
    'Clipper',
    'Clip',
    'NeedleHolder',
    'Needle',
    'Snare',
    'Suture',
    'Landmark',
    'Reflection',
    'Blood',
    'Fluid',
    'Feces',
    'PoorVisibility',
    'OutOfBody'
)

Rectal_datablocks = (ImageBlock, MaskBlock(codes=RectalCodes))
print("Number of categories:", len(RectalCodes))
print("RectalCodes:", RectalCodes)


In [ ]:
def get_y(x):
    return str(x).replace(str(IMAGES_DIR), str(MASKS_DIR))

datablock = DataBlock(
    blocks=(ImageBlock, MaskBlock(codes=RectalCodes)),
    get_items=get_image_files,
    get_y=get_y,
    item_tfms=Resize((224, 384)),
    batch_tfms=[*aug_transforms(), Normalize.from_stats(*imagenet_stats)]
)

dls = datablock.dataloaders(IMAGES_DIR, bs=4, num_workers=0)
dls.show_batch()

In [ ]:
xb, yb = dls.one_batch()
xb.shape, yb.shape

In [ ]:
yb.unique()

In [ ]:
samples = dls.decode_batch(dls.one_batch())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = min(4, len(samples))
fig, axs = plt.subplots(1, N, figsize=(3*N, 3))

for i in range(N):
    x_i, y_i = samples[i]           # x_i: PILImage, y_i: PILMask
    mask = np.array(y_i)            # [H,W] integer class ids
    bg = (mask == 0).astype(np.uint8)

    axs[i].imshow(bg, cmap='gray')  # white=background, black=non-background
    axs[i].set_title(f'Bg=0 | uniq={np.unique(mask)}')
    axs[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
samples = dls.decode_batch(dls.one_batch())

import numpy as np
import matplotlib.pyplot as plt

N = min(4, len(samples))
plt.figure(figsize=(4*N, 4))

for i in range(N):
    x_i, y_i = samples[i]

    # ---- get image as HWC uint8 ----
    x = getattr(x_i, "tensor", x_i)          # fastai TensorImage or PIL
    if hasattr(x, "permute"):                # Tensor (CHW, float)
        img = x.permute(1, 2, 0).cpu().numpy()
        if img.max() <= 1.0: img = (img*255).clip(0,255).astype(np.uint8)
        else: img = img.astype(np.uint8)
    else:                                    # PILImage
        img = np.array(x_i).astype(np.uint8)

    # ---- mask as HW ----
    m  = np.array(y_i)
    bg = (m == 0)                            # background

    # ---- overlay background in red ----
    overlay = img.copy()                     # HWC
    overlay[bg] = [255, 0, 0]                # apply on spatial dims
    alpha = 0.35
    vis = (alpha*overlay + (1-alpha)*img).astype(np.uint8)

    ax = plt.subplot(1, N, i+1)
    ax.imshow(vis)
    ax.set_title(f"uniq: {np.unique(m)}")
    ax.axis('off')

plt.tight_layout(); plt.show()

In [ ]:
len(dls.train_ds), len(dls.valid_ds)

In [ ]:
num_classes = 23

model = smp.DeepLabV3Plus(
    encoder_name='efficientnet-b5',
    encoder_weights='imagenet',
    in_channels=3,
    classes=num_classes,
    activation=None
)

In [ ]:
import sys, torch, fastai, fastcore
print("python:", sys.executable)
print("torch:", torch.__version__)
print("fastai:", fastai.__version__)
print("fastcore:", fastcore.__version__)
print("fastai path:", fastai.__file__)


In [ ]:
learn = Learner(
    dls, 
    model, 
    loss_func=FocalLossFlat(axis=1), 
    metrics=[foreground_acc, DiceMulti(axis=1)],
    opt_func=ranger
).to_fp16()

In [ ]:
import torch.serialization
from fastcore.foundation import L

torch.serialization.add_safe_globals([L])

In [ ]:
learn.to_fp32()      # TEMPORARY: disable fp16 ONLY for lr_find
learn.lr_find()     # run LR finder safely
learn.to_fp16()     # return to fp16 for training

In [ ]:
learn.fit_one_cycle(50, 1e-3)
learn.show_results(max_n=4, figsize=(6,12))

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(50, slice(1e-3/400, 1e-3/4))

In [ ]:
learn.show_results(max_n=5, figsize=(8,8))

In [ ]:
import torch, pandas as pd

def per_class_metrics(model, dl, codes):
    model.eval()
    C = len(codes)
    eps = 1e-7

    TP = torch.zeros(C, dtype=torch.float64)
    FP = torch.zeros(C, dtype=torch.float64)
    FN = torch.zeros(C, dtype=torch.float64)
    TN = torch.zeros(C, dtype=torch.float64)

    with torch.no_grad():
        for xb, yb in dl:
            preds = model(xb).argmax(1)  # [B,H,W]

            # FORCE: plain tensors on CPU (not fastai subclasses)
            preds = preds.detach().cpu().clone().to(torch.int64)
            yb    = yb.detach().cpu().clone().to(torch.int64)

            for c in range(C):
                p = (preds == c).to(torch.bool)
                g = (yb == c).to(torch.bool)

                TP[c] += (p * g).sum().item()
                FP[c] += (p * (~g)).sum().item()
                FN[c] += ((~p) * g).sum().item()
                TN[c] += ((~p) * (~g)).sum().item()

    precision = TP / (TP + FP + eps)
    recall    = TP / (TP + FN + eps)
    f1        = 2 * precision * recall / (precision + recall + eps)
    acc       = (TP + TN) / (TP + TN + FP + FN + eps)

    return pd.DataFrame({
        "Class": codes,
        "Accuracy": acc.numpy(),
        "F1": f1.numpy(),
        "Precision": precision.numpy(),
        "Recall": recall.numpy(),
    })


In [ ]:
from IPython.display import display

display(df_train)
display(df_valid)

display(df_train[df_train["Class"] != "Background"].mean(numeric_only=True))
display(df_valid[df_valid["Class"] != "Background"].mean(numeric_only=True))


In [ ]:
learn.save("deeplabv3p_effb5_final")

In [ ]:
# --- ONE-CELL SAVE (saves everything to ./models) ---

import os, json, torch

# Ensure folder exists
os.makedirs('models', exist_ok=True)

# A) Save PyTorch model weights
torch.save(learn.model.state_dict(), 'models/deeplabv3_effb5_final.pth')

# B) Save the full FastAI Learner for quick reuse
learn.export('models/deeplabv3_effb5_final.pkl')

# C) Save class labels (robust detection)
classes = None

# Try to get class names from DataLoaders
if hasattr(learn.dls, 'vocab') and learn.dls.vocab is not None:
    classes = list(learn.dls.vocab)
elif hasattr(learn.dls, 'train_ds') and hasattr(learn.dls.train_ds, 'vocab') and learn.dls.train_ds.vocab is not None:
    classes = list(learn.dls.train_ds.vocab)
elif hasattr(learn.dls, 'dataset') and hasattr(learn.dls.dataset, 'codes') and learn.dls.dataset.codes is not None:
    classes = list(learn.dls.dataset.codes)

# Fallback: infer number of classes from SMP model head
if classes is None:
    head = getattr(learn.model, 'segmentation_head', None)
    out_ch = getattr(head, 'out_channels', None)
    if out_ch is None and hasattr(head, '0') and hasattr(head[0], 'out_channels'):
        out_ch = head[0].out_channels
    if out_ch is None:
        raise RuntimeError("Could not determine number of classes; please set `classes` manually.")
    classes = [f"class_{i}" for i in range(int(out_ch))]

# Save to JSON
with open('models/deeplabv3_effb5_classes.json', 'w') as f:
    json.dump(classes, f, indent=2)

print(f"✅ Saved model files to 'models/' folder:\n"
      f" - deeplabv3_effb5_final.pth\n"
      f" - deeplabv3_effb5_final.pkl\n"
      f" - deeplabv3_effb5_classes.json ({len(classes)} classes)")
